In [ ]:
import os
import re
from ase import Atoms
from ase.units import Ry
from ase.calculators.siesta import Siesta
from ase.calculators.siesta.parameters import Species, PAOBasisBlock
from src.structure import Perovskite
from src.cleanfiles import cleanFiles
from src.fdfcreate import generate_basis

In [ ]:
perov = Perovskite('SrTiO3', bulk=True)

In [ ]:
generate_basis(perov, basis='DZPp', EnergyShift=0.001, SplitNorm=0.15, dir='resultsold/fdf')

In [ ]:
def run_test_calc(perovskite, xcf='PBEsol', basis='DZP', EnergyShift=0.01, SplitNorm=0.15,
              MeshCutoff=200, kgrid=(10, 10, 10), dir=''):


    # Define current working directory and extract information from the perovskite object
    cwd = os.getcwd()
    formula = perovskite.formula
    atoms = perovskite.atoms
    # Convert kgrid to a list to allow for modification
    kgrid = list(kgrid)

    # Calculation parameters in a dictionary
    calc_params = {
        'label': f'{formula}',
        'xc': xcf,
        'basis_set': basis,
        'mesh_cutoff': MeshCutoff * Ry,
        'energy_shift': EnergyShift * Ry,
        'kpts': kgrid,
        'directory': dir,
        'pseudo_path': os.path.join(cwd, 'pseudos', f'{xcf}')
    }
    dir_fdf = os.path.join(cwd, os.path.dirname(dir))
    # fdf arguments in a dictionary
    fdf_args = {
        '%include': os.path.join(dir_fdf, 'basis.fdf'),
        'PAO.SplitNorm': SplitNorm,
        #'SCF.DM.Tolerance': 1e-6
    }
    # Set up the Siesta calculator
    calc = Siesta(**calc_params, fdf_arguments=fdf_args)
    atoms.calc = calc
    # Perform a single-point calculation to generate the .fdf file and extract the basis block
    atoms.get_potential_energy()

    # Remove intermediate files generated by the calculation, keeping only the important ones
    cleanFiles(directory=dir, formats=['.DM', '.FA', '.XV'], confirm=False)

In [ ]:
dir = 'resultsold/fdf'
# Go up one folder from the directory
dir = os.path.dirname(dir)
dir

In [ ]:
run_test_calc(perov, xcf='PBEsol', MeshCutoff=200, kgrid=(4, 4, 4), dir='resultsold/fdf')